In [1]:
from datasets import load_dataset

from code_atlas.eval.dataset import (
    DATASET_NAME,
    DATASET_REVISION,
    HF_SPLIT,
    load_instances,
    read_split,
)

instances = load_instances()
splits = read_split()

subset = [inst for inst in instances if inst.instance_id in splits]

rows = load_dataset(DATASET_NAME, split=HF_SPLIT, revision=DATASET_REVISION)
test_patches = {r["instance_id"]: r["test_patch"] for r in rows if r["instance_id"] in splits}


/Users/andyhu/code-atlas/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def is_test_file(path: str) -> bool:
    split_path = path.split('/')
    return 'tests' in split_path[:-1]


In [3]:
cases = {
    "tests/queries/test_q.py": True,
    "sympy/core/tests/test_basic.py": True,
    "sklearn/linear_model/tests/test_ridge.py": True,
    "django/test/client.py": False,
    "foo/unittests/bar.py": False,
}
for path, expected in cases.items():
    assert is_test_file(path) == expected, path


In [4]:
from unidiff import PatchSet

def changed_paths(patch: str) -> list[str]:
    return [pf.path for pf in PatchSet(patch)]


In [5]:
patch_total = 0
wrongly_excluded = []   
for inst in subset:
    for path in changed_paths(inst.patch):
        patch_total += 1
        if is_test_file(path):
            wrongly_excluded.append((inst.instance_id, path))

wrongly_excluded


[]

In [6]:
test_patch_total = 0
wrongly_included = []   
for iid, patch in test_patches.items():
    for path in changed_paths(patch):
        test_patch_total += 1
        if not is_test_file(path):
            wrongly_included.append((iid, path))

wrongly_included

[('sympy__sympy-24102', 'sympy/testing/quality_unicode.py')]

In [7]:
print(f"(a) {len(wrongly_excluded)} / {patch_total} patch files excluded by rule")
print(f"(b) {len(wrongly_included)} / {test_patch_total} test_patch files NOT excluded by rule")


(a) 0 / 100 patch files excluded by rule
(b) 1 / 127 test_patch files NOT excluded by rule
